# 02 · Fine-tuning con QLoRA — de 33% a 89%Objetivo: un adapter entrenado, exportado y funcionando,en una T4 gratis.---### El resultado que estamos replicandoMedGemma 4B fine-tuneado sobre un dataset de MRI cerebral para clasificar cáncer:**accuracy de 33% → 89% en una sola época** (LoRA, dataset chico).Lo que **no** vamos a hacer: el notebook oficial de MedGemma con GRPO (RL) va de14,1% a 70,5%, pero pide **GPU de 40 GB y ~11 horas en A100**. Eso no es materialde fin de semana. Elegí bien dónde gastás las 48 h del hackathon.---### Presupuesto de VRAM (fuente: documentación de Unsloth)| Configuración | VRAM ||---|---|| E2B entrenar | 8 GB || E4B entrenar | 10 GB || RL | 9 GB || E4B LoRA | 17 GB || 31B QLoRA | 22 GB || 26B-A4B LoRA | **> 40 GB** ← se te escapa |**La T4 gratis de Colab (16 GB) alcanza para E4B en QLoRA.**Y una recomendación contraintuitiva de Unsloth: **es mejor entrenar E4B en QLoRA queE2B en LoRA**. El E4B es más grande y la diferencia por cuantizar es despreciable.

In [ ]:
!pip install -q unsloth!pip install -q --no-deps trl peft accelerate bitsandbytes!nvidia-smi --query-gpu=name,memory.total --format=csv    

## Las cuatro trampasAntes de escribir una línea, memorizá estas cuatro. Las vas a cometer a las 3 AM.1. **El MoE se lleva mal con 4 bits.** El routing del 26B-A4B y la cuantización   4-bit interactúan pésimo → para el MoE usá **LoRA 16-bit**, no QLoRA.2. **El rol se llama `model`, no `assistant`.** Silencioso y letal.3. **Perdiste el razonamiento** → mezclá **≥75% de ejemplos con reasoning**.   En multi-turno, en el target va **solo la respuesta final visible**, no los   bloques de pensamiento previos.4. **Anda en Colab, se rompe en Ollama** → **chat template mismatch**. Causa #1.**Bonus gratis:** si ves un **loss de 13–15** en E2B/E4B, **es normal**. Quirk conocidode los multimodales. No lo arregles.

In [ ]:
from unsloth import FastModelimport torchMODEL = "unsloth/gemma-4-E4B-it"   # <-- verificar el ID exacto en unsloth.ai/docsMAX_SEQ = 2048model, tokenizer = FastModel.from_pretrained(    model_name = MODEL,    max_seq_length = MAX_SEQ,    load_in_4bit = True,      # QLoRA. Para el MoE 26B-A4B: poner False y load_in_16bit=True    full_finetuning = False,)    

## Configuración de LoRAHiperparámetros de arranque razonables. `alpha = rank` mantiene el learning rateefectivo; `alpha = 2*rank` hace el adapter más fuerte. Subí a rango 32 o 64 si tudominio está lejos del pretraining (la terminología periodontal lo está; escribiremails no).

In [ ]:
model = FastModel.get_peft_model(    model,    r = 16,                # 8-64. Empezá en 16.    lora_alpha = 16,       # alpha = r  -> LR efectivo sin cambios    lora_dropout = 0,      # Unsloth recomienda 0    bias = "none",    target_modules = [        "q_proj", "k_proj", "v_proj", "o_proj",     # atención        "gate_proj", "up_proj", "down_proj",        # feed-forward    ],    use_gradient_checkpointing = "unsloth",    random_state = 3407,)trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)total = sum(p.numel() for p in model.parameters())print(f"Entrenables: {trainable:,} / {total:,}  ({100*trainable/total:.2f}%)")# Debería dar entre 0.2% y 1%. Si da mucho más, algo está mal.    

## Los datos — el 80% del trabajo realTamaños realistas:- **200–1.000** ejemplos para transferencia de estilo/formato- **500–5.000** para clasificación**El error #1 que busca el jurado: data leakage a nivel de paciente.**Si dos cortes del mismo paciente caen uno en train y otro en test, tu accuracy estáinflada y **no lo sabés**. El pipeline oficial de Google Health para histopatologíahace **patient-level splitting** justamente por esto.Reemplazá `armar_dataset()` con tus datos. La estructura del mensaje es lo que importa.

In [ ]:
from datasets import Datasetimport randomdef armar_dataset(n=600):    """PLACEHOLDER. Reemplazar con datos reales.    Tarea de juguete: extraer datos estructurados de un informe de laboratorio.    Es la misma forma que tendría el problema real (texto -> JSON estricto).    """    analitos = [("Glucosa","mg/dL",70,110), ("Creatinina","mg/dL",0.6,1.3),                ("Hemoglobina","g/dL",12,17), ("Potasio","mEq/L",3.5,5.1)]    filas = []    for _ in range(n):        nombre, unidad, lo, hi = random.choice(analitos)        valor = round(random.uniform(lo*0.5, hi*1.6), 2)        estado = "bajo" if valor < lo else ("alto" if valor > hi else "normal")        informe = f"LABORATORIO CENTRAL\n{nombre}: {valor} {unidad}  (VR {lo}-{hi})"        salida = f'{{"analito": "{nombre}", "valor": {valor}, "unidad": "{unidad}", "estado": "{estado}"}}'        filas.append({"informe": informe, "json": salida})    return filasSYSTEM = ("Extraés datos de informes de laboratorio y devolvés únicamente JSON. "          "Nunca inventás valores: si un campo no está, ponés null.")def a_conversacion(row):    return {"messages": [        {"role": "system",  "content": SYSTEM},        {"role": "user",    "content": row["informe"]},        # 🔴 GEMMA 4 USA "model", NO "assistant". Esta línea es la trampa #2.        {"role": "model",   "content": row["json"]},    ]}filas = armar_dataset()ds = Dataset.from_list([a_conversacion(r) for r in filas])split = ds.train_test_split(test_size=0.15, seed=42)   # ← en datos médicos: split POR PACIENTEtrain_ds, eval_ds = split["train"], split["test"]print(train_ds[0]["messages"])    

In [ ]:
from unsloth.chat_templates import get_chat_template# 🔴 Trampa #4: el template que usás para entrenar tiene que ser el mismo que usa# el runtime de inferencia. Guardá este nombre y verificalo al exportar.tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")def formatear(ejemplos):    textos = [        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)        for m in ejemplos["messages"]    ]    return {"text": textos}train_ds = train_ds.map(formatear, batched=True)eval_ds  = eval_ds.map(formatear, batched=True)print(train_ds[0]["text"][:400])    

## Baseline ANTES de entrenar**Sin baseline no hay resultado.** Vale el 30% del puntaje del jurado.Medí primero. Siempre.

In [ ]:
import json, refrom unsloth import FastModeldef predecir(modelo, tok, informe):    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": informe}]    inputs = tok.apply_chat_template(        msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True    ).to(modelo.device)    out = modelo.generate(**inputs, max_new_tokens=128, do_sample=False)    return tok.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)def evaluar(modelo, tok, dataset, n=60):    ok_json, ok_exacto = 0, 0    for row in dataset.select(range(min(n, len(dataset)))):        informe  = row["messages"][1]["content"]        esperado = json.loads(row["messages"][2]["content"])        crudo = predecir(modelo, tok, informe)        crudo = re.sub(r"^```(?:json)?|```$", "", crudo.strip(), flags=re.M).strip()        try:            pred = json.loads(crudo)            ok_json += 1            if pred == esperado:                ok_exacto += 1        except json.JSONDecodeError:            pass    total = min(n, len(dataset))    return {"json_valido": ok_json/total, "match_exacto": ok_exacto/total, "n": total}FastModel.for_inference(model)baseline = evaluar(model, tokenizer, eval_ds)print("BASELINE:", baseline)    

## Entrenar

In [ ]:
from trl import SFTTrainer, SFTConfigfrom unsloth import FastModelFastModel.for_training(model)trainer = SFTTrainer(    model = model,    tokenizer = tokenizer,    train_dataset = train_ds,    dataset_text_field = "text",    max_seq_length = MAX_SEQ,    args = SFTConfig(        per_device_train_batch_size = 2,        gradient_accumulation_steps = 4,   # batch efectivo = 8        warmup_ratio = 0.03,        num_train_epochs = 1,              # empezá con 1. En serio.        learning_rate = 2e-4,        max_grad_norm = 0.3,        lr_scheduler_type = "linear",        logging_steps = 10,        optim = "adamw_8bit",        seed = 3407,        output_dir = "outputs",        report_to = "none",    ),)stats = trainer.train()    

### Sobre el lossSi ves **13–15** en E2B/E4B: **es normal**. Quirk de los multimodales. No lo arregles.Lo que importa no es el loss. Es la métrica de la tarea. Que es lo que sigue.

In [ ]:
FastModel.for_inference(model)despues = evaluar(model, tokenizer, eval_ds)print(f"BASELINE  json_valido={baseline['json_valido']:.1%}  match={baseline['match_exacto']:.1%}")print(f"DESPUÉS   json_valido={despues['json_valido']:.1%}  match={despues['match_exacto']:.1%}")print(f"\nΔ match exacto: {(despues['match_exacto']-baseline['match_exacto'])*100:+.1f} puntos")    

## Exportar — donde se rompe todo**Trampa #4.** El adapter pesa ~30–100 MB a rango 16. El base no se toca.Guardamos **solo el adapter** — que es lo que hace posible el notebook 03.

In [ ]:
# Solo el adapter (~60 MB). Esto es lo que se sirve con vLLM multi-LoRA.model.save_pretrained("adapter_labs")tokenizer.save_pretrained("adapter_labs")!du -sh adapter_labs!ls adapter_labs# Tiene que haber adapter_config.json + adapter_model.safetensors:# son los dos archivos que vLLM necesita para cargarlo.    

In [ ]:
# Opcional: mergear y exportar a GGUF para Ollama.# ⚠️ Después de esto, VERIFICÁ que el chat template del GGUF sea el mismo que usaste# para entrenar. Si el modelo responde raro en Ollama y bien acá, es esto.## model.save_pretrained_gguf("gemma4_labs_gguf", tokenizer, quantization_method="q4_k_m")    

## Checklist antes de presentar tu proyecto- [ ] Hay un **baseline** medido antes de entrenar.- [ ] El split es **por paciente**, no por imagen.- [ ] La métrica es la correcta para el desbalance (F1, no accuracy).- [ ] Probaste el modelo exportado **en el runtime de destino**, no solo en el notebook.- [ ] Sabés **dónde falla** y me lo podés mostrar.